<a href="https://colab.research.google.com/github/MohamadHammoura/CRUD_Cars_Homework/blob/main/Unit_8_Lab_Threat_Hunter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 8: Threat Hunter

---
---

## Overview
In this lab we will be completing two guided hunts from the Threat Hunter Playbook. For each hunt we will be using the provided security dataset to investigate what the packets look like from a real attack! These datasets follow the MITRE ATT&CK structure for categorizing post-compromise adversary behavior. Security professionals use open-source resources like this to prepare for future attacks by finding patterns among past exploits, and we will be doing the same!

### Goals
*   Query the LSASS Memory Read Access dataset and analyze the corresponding outputs.
*   Query the PowerShell Remote Session dataset and analyze the corresponding outputs.
*   Gain a greater understanding of how adversaries aim to compromise systems, and how to analyze when they've done so.

### Resources
*   [The Threat Hunter Playbook](https://github.com/OTRF/ThreatHunter-Playbook) (GitHub)
*   [The Threat Hunter Playbook](https://threathunterplaybook.com/intro.html) (docs)
*   [What is PowerShell?](https://learn.microsoft.com/en-us/powershell/scripting/overview?view=powershell-7.3)

---

## Lab Instructions

### HUNT 1: [LSASS Memory Read Access](https://threathunterplaybook.com/hunts/windows/170105-LSASSMemoryReadAccess/notebook.html)

LSASS (Local Security Authority Subsystem Service) maintains the security policy for a system. This includes logging on users, password changes, and creating security credentials for a user session (access tokens). The file path and name for this process is **C:\windows\system32\lsass.exe**.

With such sensitive information being stored here, **lsass.exe** makes a great target for adversaries. This [article](https://redcanary.com/threat-detection-report/techniques/lsass-memory/) has a lot of great information, including how adversaries aim to compromise LSASS memory, and how we can detect when they do.

Below, we will be running a few analytics on this [dataset](https://securitydatasets.com/notebooks/atomic/windows/credential_access/SDWIN-190518202151.html) to determine how and where the adversary extracted private credentials from memory.

In [ ]:
import requests
from zipfile import ZipFile
from io import BytesIO

url = 'https://raw.githubusercontent.com/OTRF/Security-Datasets/master/datasets/atomic/windows/credential_access/host/empire_mimikatz_logonpasswords.zip'
zipFileRequest = requests.get(url)
zipFile = ZipFile(BytesIO(zipFileRequest.content))
datasetJSONPath = zipFile.extract(zipFile.namelist()[0])

In [ ]:
import pandas as pd
from pandas.io import json

df = json.read_json(path_or_buf=datasetJSONPath, lines=True)

#### Analytic I

**Event 4656:** A handle to an object was requested

**Event 4663:** An attempt was made to access an object

The three main parts of our output we want to focus on for this analytic are the ProcessName, ObjectName, and Event ID. We can conclude from the ProcessName that some PowerShell script (written by a user) has attempted to access the **lsass.exe** process (gathered from the ObjectName). Whenever something besides a System process accesses **lsass.exe** we want to be very suspicious, since the user has essentially no reason to interact with it.

In [ ]:
(
df[['@timestamp','Hostname','SubjectUserName','ProcessName','ObjectName','AccessMask','EventID']]

[(df['Channel'].str.lower() == 'security')
    & ((df['EventID'] == 4663) | (df['EventID'] == 4656))
    & (df['ObjectName'].str.contains('.*lsass.exe', regex=True))
    & (~df['SubjectUserName'].str.endswith('.*$', na=False))
]
.head()
)

,@timestamp,Hostname,SubjectUserName,ProcessName,ObjectName,AccessMask,EventID
2459,2020-08-07T14:32:57.592Z,WORKSTATION5.theshire.local,pgustavo,C:\Windows\System32\WindowsPowerShell\v1.0\pow...,\Device\HarddiskVolume2\Windows\System32\lsass...,0x1010,4656
2460,2020-08-07T14:32:57.592Z,WORKSTATION5.theshire.local,pgustavo,C:\Windows\System32\WindowsPowerShell\v1.0\pow...,\Device\HarddiskVolume2\Windows\System32\lsass...,0x10,4663


#### Analytic II

Identify what the Event ID indicates by using this [site](https://www.ultimatewindowssecurity.com/securitylog/encyclopedia/default.aspx?i=j). On the left side of the page under where it says "Go To Event ID:", input the Event ID you would like to know more about (in this case input 10).

For this particular event, all that the site states is that this is a Sysmon (System Monitoring) event. Sysmon logs system activity, and the **ProcessAccess** event in particular will log when a process opens another process. This is precisely what we are looking for as an example of how adversaries can read the memory of **lsass.exe**, which we can see is the TargetImage in our output.

One more thing to note is the involvement of DLLs (Dynamic Link Libraries). These are files containing code and data that programs can use as a knowledge base to perform certain functions. This short [video](https://youtu.be/5vLaqjR7DKI?t=167) provides some more information on DLLs, but the importance of DLLs in this analytic has to do with the CallTrace column in our output. As we stated, Sysmon events such as **ProcessAccess** produce logs of system activity. The CallTrace field will log the DLL used during the process in question. We see it is "UNKNOWN" for this process, adding to our suspicion.

In [ ]:
(
df[['@timestamp','Hostname','SourceImage','TargetImage','GrantedAccess','SourceProcessGUID','CallTrace']]

[(df['Channel'] == 'Microsoft-Windows-Sysmon/Operational')
    & (df['EventID'] == 10)
    & (df['TargetImage'].str.contains('.*lsass.exe', regex=True))
    & (df['CallTrace'].str.contains('.*UNKNOWN*', regex=True))
]
.head()
)

,@timestamp,Hostname,SourceImage,TargetImage,GrantedAccess,SourceProcessGUID,CallTrace
2450,2020-08-07T14:32:57.589Z,WORKSTATION5.theshire.local,C:\windows\System32\WindowsPowerShell\v1.0\pow...,C:\windows\system32\lsass.exe,0x1010,{297bc33e-65d0-5f2d-8207-000000000400},C:\windows\SYSTEM32\ntdll.dll+9c534|C:\windows...


#### Analytic III

Look up Event 7, and we should see that this is the image loaded event. This event logs which DLLs are loaded when processes are run, a more general way of viewing DLL usage which cannot be hidden.

This is important because adversaries need to employ certain DLLs to retrieve the security information from **lsass.exe**. Perhaps the most popular example of this is [Mimikatz](https://github.com/gentilkiwi/mimikatz), an open-source program that to this day is an effective tool used to dump sensitive credentials, but is more commonly used to test a system's defenses.

Our output details that out of the 5 DLLs known to be used by Mimikatz, 4 of them were loaded by the adversary-run process (**powershell.exe** in the Image column). Analyzing DLL usage is a great way of discovering what adveraries are doing even when they attempt to cover their tracks.

In [ ]:
(
df[['@timestamp','ProcessGuid','Image','ImageLoaded']]

[(df['Channel'] == 'Microsoft-Windows-Sysmon/Operational')
    & (df['EventID'] == 7)
    & (
        (df['ImageLoaded'].str.contains('.*samlib.dll', regex=True))
        | (df['ImageLoaded'].str.contains('.*vaultcli.dll', regex=True))
        | (df['ImageLoaded'].str.contains('.*hid.dll', regex=True))
        | (df['ImageLoaded'].str.contains('.*winscard.dll', regex=True))
        | (df['ImageLoaded'].str.contains('.*cryptdll.dll', regex=True))
    )
    & (df['@timestamp'].between('2020-06-00 00:00:00.000','2020-08-20 00:00:00.000'))
]
.groupby(['ProcessGuid','Image'])['ImageLoaded'].count().sort_values(ascending=False)
).to_frame()

,,ImageLoaded
ProcessGuid,Image,
{297bc33e-65d0-5f2d-8207-000000000400},C:\Windows\System32\WindowsPowerShell\v1.0\powershell.exe,4


#### Analytic IV

This analytic provides a summarized output of what we have discovered thus far - a PowerShell process has attempted to access the **lsass.exe** process by loading 4 of 5 DLLs used by Mimikatz, a program used to dump user credentials.

With this information, we have successfully analyzed some very suspicious, and likely malicious adversary activity.

In [ ]:
imageLoadDf = (
df[['@timestamp','ProcessGuid','Image','ImageLoaded']]

[(df['Channel'] == 'Microsoft-Windows-Sysmon/Operational')
    & (df['EventID'] == 7)
    & (
        (df['ImageLoaded'].str.contains('.*samlib.dll', regex=True))
        | (df['ImageLoaded'].str.contains('.*vaultcli.dll', regex=True))
        | (df['ImageLoaded'].str.contains('.*hid.dll', regex=True))
        | (df['ImageLoaded'].str.contains('.*winscard.dll', regex=True))
        | (df['ImageLoaded'].str.contains('.*cryptdll.dll', regex=True))
    )
    & (df['@timestamp'].between('2020-06-00 00:00:00.000','2020-08-20 00:00:00.000'))
]
.groupby(['ProcessGuid','Image'])['ImageLoaded'].count().sort_values(ascending=False)
).to_frame()

processAccessDf = (
df[['@timestamp', 'Hostname', 'SourceImage', 'TargetImage', 'GrantedAccess', 'SourceProcessGUID']]

[(df['Channel'] == 'Microsoft-Windows-Sysmon/Operational')
    & (df['EventID'] == 10)
    & (df['TargetImage'].str.contains('.*lsass.exe', regex=True))
    & (df['CallTrace'].str.contains('.*UNKNOWN*', regex=True))
]
)

(
pd.merge(imageLoadDf, processAccessDf,
    left_on = 'ProcessGuid', right_on = 'SourceProcessGUID', how = 'inner')
)

,ImageLoaded,@timestamp,Hostname,SourceImage,TargetImage,GrantedAccess,SourceProcessGUID
0,4,2020-08-07T14:32:57.589Z,WORKSTATION5.theshire.local,C:\windows\System32\WindowsPowerShell\v1.0\pow...,C:\windows\system32\lsass.exe,0x1010,{297bc33e-65d0-5f2d-8207-000000000400}


### Hunt 2: [PowerShell Remote Session](https://threathunterplaybook.com/hunts/windows/190511-RemotePwshExecution/notebook.html)

PowerShell is similar in appearance to the command-line shell, but it is actually a much more powerful framework (frequently used by system administrators for this reason). This [article](https://redcanary.com/threat-detection-report/techniques/powershell/) has a lot of useful information detailing how adversaries can use this powerful framework against us.

Below, you will be tasked with running analytics on this [dataset](https://securitydatasets.com/notebooks/atomic/windows/execution/SDWIN-190518211456.html), and interpreting the outputs just like we did in the previous section. This can be tricky to do alone, so be sure to follow along with the hunt guide (linked in the title of this section) for hints on what each analytic is doing! It is also a good idea to look up the [Event IDs](https://www.ultimatewindowssecurity.com/securitylog/encyclopedia/default.aspx?i=j) for each analytic for a more detailed understanding of what's happening.

In [ ]:
import requests
from zipfile import ZipFile
from io import BytesIO

url = 'https://raw.githubusercontent.com/OTRF/Security-Datasets/master/datasets/atomic/windows/lateral_movement/host/empire_psremoting_stager.zip'
zipFileRequest = requests.get(url)
zipFile = ZipFile(BytesIO(zipFileRequest.content))
datasetJSONPath = zipFile.extract(zipFile.namelist()[0])

In [ ]:
import pandas as pd
from pandas.io import json

df = json.read_json(path_or_buf=datasetJSONPath, lines=True)

#### Analytic I

In [ ]:
(
df[['@timestamp','Hostname','Channel']]

[(df['Channel'].isin(['Microsoft-Windows-PowerShell/Operational','Windows PowerShell']))
    & (df['EventID'].isin([400,4103]))
    & (df['Message'].str.contains('.*HostApplication.*wsmprovhost.*', regex=True))
]
.head()
)

,@timestamp,Hostname,Channel
678,2020-09-20T21:09:11.871Z,WORKSTATION6,Windows PowerShell


#### Analytic II

In [ ]:
(
df[['@timestamp','Hostname','Application','SourceAddress','DestAddress','LayerName','LayerRTID']]

[(df['Channel'].str.lower() == 'security')
    & (df['EventID'] == 5156)
    & (
        (df['DestPort'] == 5985)
        | (df['DestPort'] == 5986)
    )
    & (df['LayerRTID'] == 44)
]
.head()
)

,@timestamp,Hostname,Application,SourceAddress,DestAddress,LayerName,LayerRTID
478,2020-09-20T21:09:09.800Z,WORKSTATION6.theshire.local,System,172.18.39.5,172.18.39.6,%%14610,44.0
745,2020-09-20T21:09:11.902Z,WORKSTATION6.theshire.local,System,172.18.39.5,172.18.39.6,%%14610,44.0
856,2020-09-20T21:09:12.962Z,WORKSTATION6.theshire.local,System,172.18.39.5,172.18.39.6,%%14610,44.0


#### Analytic III

In [ ]:
(
df[['@timestamp','Hostname','ParentProcessName','NewProcessName']]

[(df['Channel'].str.lower() == 'security')
    & (df['EventID'] == 4688)
    & (
        (df['ParentProcessName'].str.lower().str.endswith('wsmprovhost.exe', na=False))
        | (df['NewProcessName'].str.lower().str.endswith('wsmprovhost.exe', na=False))
    )
]
.head()
)

,@timestamp,Hostname,ParentProcessName,NewProcessName
535,2020-09-20T21:09:09.825Z,WORKSTATION6.theshire.local,C:\Windows\System32\svchost.exe,C:\Windows\System32\wsmprovhost.exe
867,2020-09-20T21:09:12.968Z,WORKSTATION6.theshire.local,C:\Windows\System32\wsmprovhost.exe,C:\Windows\System32\WindowsPowerShell\v1.0\pow...


#### Analytic IV

In [ ]:
(
df[['@timestamp','Hostname','ParentImage','Image']]

[(df['Channel'] == 'Microsoft-Windows-Sysmon/Operational')
    & (df['EventID'] == 1)
    & (
        (df['ParentImage'].str.lower().str.endswith('wsmprovhost.exe', na=False))
        | (df['Image'].str.lower().str.endswith('wsmprovhost.exe', na=False))
    )
]
.head()
)

,@timestamp,Hostname,ParentImage,Image
516,2020-09-20T21:09:09.817Z,WORKSTATION6.theshire.local,C:\Windows\System32\svchost.exe,C:\Windows\System32\wsmprovhost.exe
843,2020-09-20T21:09:12.956Z,WORKSTATION6.theshire.local,C:\Windows\System32\wsmprovhost.exe,C:\Windows\System32\WindowsPowerShell\v1.0\pow...


#### Analytic V

In [ ]:
(
df[['@timestamp','Hostname','User','Initiated','Image','SourceIp','DestinationIp']]

[(df['Channel'] == 'Microsoft-Windows-Sysmon/Operational')
    & (df['EventID'] == 3)
    & (
        (df['DestinationPort'] == 5985)
        | (df['DestinationPort'] == 5986)
    )
    & (~df['User'].isin(['NT AUTHORITY\\NETWORK SERVICE', 'NT AUTHORITY\\SYSTEM']))
]
)

,@timestamp,Hostname,User,Initiated,Image,SourceIp,DestinationIp
689,2020-09-20T21:09:11.877Z,WORKSTATION5.theshire.local,THESHIRE\pgustavo,true,C:\Windows\System32\WindowsPowerShell\v1.0\pow...,172.18.39.5,172.18.39.6
1415,2020-09-20T21:09:13.263Z,WORKSTATION5.theshire.local,THESHIRE\pgustavo,true,C:\Windows\System32\WindowsPowerShell\v1.0\pow...,172.18.39.5,172.18.39.6
1418,2020-09-20T21:09:13.264Z,WORKSTATION5.theshire.local,THESHIRE\pgustavo,true,C:\Windows\System32\WindowsPowerShell\v1.0\pow...,172.18.39.5,172.18.39.6


In [ ]:
df.groupby(['Channel']).size().sort_values(ascending=False)

,0
Channel,
Microsoft-Windows-Sysmon/Operational,1651
security,544
Windows PowerShell,217
Microsoft-Windows-PowerShell/Operational,173
Security,152
System,6
Microsoft-Windows-WMI-Activity/Operational,1
